In [2]:
import torch
import torch.nn as nn
import pandas as pd
import joblib

In [19]:
df = pd.read_csv(r"dataset/test.csv")

In [20]:
df.head(3)

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,750000,male,45,177.0,81.0,7.0,87.0,39.8
1,750001,male,26,200.0,97.0,20.0,101.0,40.5
2,750002,female,29,188.0,85.0,16.0,102.0,40.4


In [54]:
df_2 = pd.DataFrame({
    "id":1,
    "Sex":"male",
    "Age":36,
    "Height":189,
    "Weight":82,
    "Duration":26,
    "Heart_Rate":101,
    "Body_Temp":41.0
},index=[0])

In [55]:
df_2

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,1,male,36,189,82,26,101,41.0


In [56]:
onehotencoder = joblib.load('OneHotEncoder.pkl')
x_train_scaler = joblib.load("x_train_x_test.pkl")
y_train_scaler = joblib.load("y_train_y_test.pkl")

In [57]:
encoder = onehotencoder.transform(df_2[["Sex"]])

In [58]:
column_name_encoder = onehotencoder.get_feature_names_out()
column_name_encoder

array(['Sex_male'], dtype=object)

In [59]:
encoder_df = pd.DataFrame(encoder,columns=column_name_encoder)
encoder_df

,Sex_male
0,1.0


In [60]:
final_df = pd.concat(
    [df_2.drop("Sex",axis=1),encoder_df],axis=1
)

In [61]:
final_df

,id,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Sex_male
0,1,36,189,82,26,101,41.0,1.0


In [62]:
x = final_df.values

In [63]:
x_train_scaler = x_train_scaler.transform(x)

In [64]:
x_train = torch.tensor(x_train_scaler,dtype=torch.float32)

In [65]:
model = nn.Sequential(
    nn.Linear(8,128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(128,64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(64,32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(32,1)
)

In [66]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

In [67]:
checkpoint = torch.load(
    r"Checkpoints_199.pth"
)

In [68]:
checkpoint

{'epochs': 199,
 'model': OrderedDict([('0.weight',
               tensor([[-0.1018, -0.2508, -0.0261,  ...,  0.2529,  0.0248, -0.1568],
                       [-0.0951, -0.0899, -0.2691,  ..., -0.0215,  0.0514,  0.1572],
                       [ 0.1896, -0.0839,  0.3034,  ..., -0.2331, -0.3378, -0.0800],
                       ...,
                       [ 0.2884, -0.3679,  0.3106,  ..., -0.1817, -0.2432,  0.1709],
                       [-0.1219, -0.2678,  0.1088,  ...,  0.0463,  0.1226, -0.0814],
                       [ 0.1454, -0.0678,  0.0723,  ...,  0.4513,  0.3182,  0.0077]])),
              ('0.bias',
               tensor([-0.1458,  0.1251, -0.1538, -0.1734,  0.2184,  0.1117,  0.2345,  0.2764,
                       -0.1836, -0.0100, -0.2883, -0.2933, -0.3149, -0.0022, -0.0704,  0.2733,
                       -0.2243, -0.2303,  0.2656,  0.1523, -0.0071, -0.0158,  0.1200,  0.0534,
                       -0.2386,  0.0858, -0.0827,  0.0265,  0.2163,  0.0912,  0.0694, -0.1523,
  

In [69]:
model.load_state_dict(checkpoint["model"])

<All keys matched successfully>

In [70]:
optimizer.load_state_dict(checkpoint["optimizer_function"])

In [71]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [72]:
epochs = checkpoint["epochs"]

In [73]:
epochs

199

In [74]:
loss = checkpoint["loss"]

In [75]:
loss

0.0555766262114048

In [76]:
model.eval()

Sequential(
  (0): Linear(in_features=8, out_features=128, bias=True)
  (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (6): ReLU()
  (7): Dropout(p=0.2, inplace=False)
  (8): Linear(in_features=64, out_features=32, bias=True)
  (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (10): ReLU()
  (11): Dropout(p=0.2, inplace=False)
  (12): Linear(in_features=32, out_features=1, bias=True)
)

In [77]:
with torch.no_grad():
    pred = model(x_train)

pred = y_train_scaler.inverse_transform(pred)

In [81]:
int(pred.round())

/tmp/ipykernel_3548/715960600.py:1: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  int(pred.round())


144